# 03 — Tier-1 MCTS-HNM Benchmarks: Multi-Seed Baselines, Hierarchy Ablations & Sensitivity

**Feature**: `F4` (`04-tier1-benchmark-suite` | `EPIC-04`)  
**Architecture Decisions**: `AD-001` (Tier-1 Embedding Protocol), `AD-002` (FastHierarchyTree Protocol), `AD-004` (Scientific Provenance)  

This notebook runs the comprehensive Tier-1 benchmark suite evaluating the MCTS-HNM framework:
1. **Axis 1 (Sampler Comparison)**: Evaluates Uniform, OHEM, Strict MCTS, and Enhanced MCTS across imbalance ratios (Scenario 1, One-vs-Rest).
2. **Axis 2 (Hierarchy Geometry Ablation)**: Compares Semantic WordNet hierarchy vs. Ward confusion tree clustering.
3. **Scenario 2 (Multi-Class Curriculum)**: Evaluates 100-way multi-class classification accuracy, macro-F1, and worst-10-class recall under MCTS curriculum.
4. **Axis 3 (Hyperparameter Sensitivity)**: Grid ablation over exploration parameter $c \in \{0.5, 1.0, 2.0, 4.0\}$ and momentum $\alpha \in \{0.1, 0.3, 0.5, 0.7\}$.
5. **Real MCTS Tree Visitation Diagnostics**: Instruments and persists real node visitation counters and superclass distribution to `tier1_visitation_diagnostics.json`.


In [1]:
from pathlib import Path
import os
import sys

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from src.utils.experiment_config import (
    ExecutionMode,
    ExperimentConfig,
    get_isolated_paths,
    print_execution_banner,
)
from src.data.cifar100_taxonomy import get_cifar100_taxonomy
from src.data.extract_embeddings import load_embedding_cache
from src.training.benchmark_runner import (
    audit_scenario1_imbalance,
    run_benchmark,
    run_sensitivity_ablation,
)
from src.utils.seed import set_global_seed

# Execution mode: REAL by default for scientific runs; SMOKE for testing
EXECUTION_MODE = ExecutionMode.REAL
if os.environ.get("EXECUTION_MODE"):
    EXECUTION_MODE = ExecutionMode(os.environ["EXECUTION_MODE"].lower())
elif "pytest" in sys.modules:
    EXECUTION_MODE = ExecutionMode.SMOKE

config = ExperimentConfig(execution_mode=EXECUTION_MODE)
print_execution_banner(config, notebook_name="03_tier1_mcts_benchmarks")
paths = get_isolated_paths(config, repo_root=REPO_ROOT)

SEED = set_global_seed(42)
RESULTS_DIR = paths.results_dir
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Repository root: {REPO_ROOT} | Results directory: {RESULTS_DIR}")


EXPERIMENT MODE: REAL
Dataset: CIFAR-100
Synthetic fallback: DISABLED
Scientific results: ENABLED
Notebook: 03_tier1_mcts_benchmarks
Mode: REAL
Output directory: /home/mlima/workspace/mcts-hnm/results/real
Repository root: /home/mlima/workspace/mcts-hnm | Results directory: /home/mlima/workspace/mcts-hnm/results/real


## 1. Load Precomputed ResNet-18 Embeddings

Loads `embeddings/cifar100_resnet18.pt` generated by Notebook 01, or synthesizes a deterministic fixture if cache is absent.

In [2]:
CACHE_PATH = paths.embeddings_dir / "cifar100_resnet18.pt"

if not CACHE_PATH.is_file():
    if config.execution_mode == ExecutionMode.REAL:
        raise FileNotFoundError(
            f"Real CIFAR-100 embeddings not found at {CACHE_PATH}. Run Notebook 01 in REAL mode first."
        )
    else:
        from src.data.extract_embeddings import extract_or_generate_embeddings
        extract_or_generate_embeddings(
            output_path=CACHE_PATH,
            config=config,
            smoke_samples_per_class=30,
            smoke_feature_dim=64,
        )

bundle = load_embedding_cache(CACHE_PATH)
train_features = bundle["train"]["embeddings"]
train_labels = bundle["train"]["fine_labels"]
val_features = bundle["val"]["embeddings"]
val_labels = bundle["val"]["fine_labels"]
test_features = bundle["test"]["embeddings"]
test_labels = bundle["test"]["fine_labels"]
print(f"Loaded {config.execution_mode.value.upper()} embeddings: Train={tuple(train_features.shape)}, Val={tuple(val_features.shape)}, Test={tuple(test_features.shape)}")


Loaded REAL embeddings: Train=(45000, 512), Val=(5000, 512), Test=(10000, 512)


## 2. Axis 1: Sampler Comparison in Scenario 1 (Uniform vs. OHEM vs. Strict vs. Enhanced MCTS)

Compares the four negative sampling strategies across multiple seeds on Scenario 1 (One-vs-Rest, target class `0` / `beaver`, imbalanced positive pool).

Evaluates the canonical imbalance ratio $\rho = 1:50$ by default, subsampling positive instances via `src.data.splits.build_scenario1_split()`. For full publication deliverables, `imbalance_ratios` can be expanded to `("1:10", "1:50", "1:100")`.

In [3]:
SEEDS = (42, 43, 44, 45, 46) if config.execution_mode == ExecutionMode.REAL else (42,)
num_steps = 1000 if config.execution_mode == ExecutionMode.REAL else 10
eval_every = 25 if config.execution_mode == ExecutionMode.REAL else 5
IMBALANCE_RATIOS_AXIS1 = ("1:50",)  # Evaluated canonical ratio; can be expanded to ("1:10", "1:50", "1:100") for full grid

res_axis1 = run_benchmark(
    scenario="scenario1",
    train_features=train_features,
    train_labels=train_labels,
    test_features=test_features,
    test_labels=test_labels,
    strategies=("uniform", "ohem", "mcts_strict", "mcts_enhanced"),
    tree_types=("semantic",),
    seeds=SEEDS,
    output_dir=RESULTS_DIR,
    benchmark_name="tier1_axis1_samplers",
    num_steps=num_steps,
    eval_every=eval_every,
    target_class_scenario1=0,
    val_features=val_features,
    val_labels=val_labels,
    config=config,
    imbalance_ratios=IMBALANCE_RATIOS_AXIS1,
)

df_axis1 = pd.read_csv(RESULTS_DIR / "tier1_axis1_samplers.csv")
summary_axis1 = df_axis1.groupby("strategy")[["auprc", "f1", "fpr_at_95_recall"]].agg(["mean", "std"])
print("=== Axis 1: Sampler Comparison Results (Scenario 1) ===")
print(summary_axis1)
if res_axis1.get("statistical_comparisons"):
    print("\nStatistical tests against Enhanced MCTS-HNM:")
    for comp, stats in res_axis1["statistical_comparisons"].items():
        print(f"  {comp}: diff={stats['mean_diff']:.4f}, t_p={stats['t_pvalue']:.4e}, w_p={stats['wilcoxon_pvalue']:.4e}")


=== Axis 1: Sampler Comparison Results (Scenario 1) ===
                  auprc                  f1           fpr_at_95_recall  \
                   mean       std      mean       std             mean   
strategy                                                                 
mcts_enhanced  0.684816  0.022518  0.523994  0.094018         0.355737   
mcts_strict    0.735776  0.016832  0.630517  0.052371         0.240303   
ohem           0.703195  0.012520  0.510865  0.036686         0.326364   
uniform        0.707786  0.038021  0.608050  0.071040         0.282202   

                         
                    std  
strategy                 
mcts_enhanced  0.169167  
mcts_strict    0.116565  
ohem           0.176571  
uniform        0.133730  

Statistical tests against Enhanced MCTS-HNM:
  mcts_enhanced_semantic_vs_uniform_none: diff=-0.0230, t_p=1.5797e-01, w_p=1.8750e-01
  mcts_enhanced_semantic_vs_ohem_none: diff=-0.0184, t_p=2.8253e-02, w_p=6.2500e-02
  mcts_enhanced_semantic_v

## 3. Axis 2: Hierarchy Geometry Ablation (Semantic vs. Ward Confusion Tree)

Tests whether empirical visual error clustering (Ward link on validation confusion matrix) improves MCTS mining efficiency over static taxonomy.

In [4]:
res_axis2 = run_benchmark(
    scenario="scenario1",
    train_features=train_features,
    train_labels=train_labels,
    test_features=test_features,
    test_labels=test_labels,
    strategies=("mcts_enhanced",),
    tree_types=("semantic", "confusion_ward"),
    seeds=SEEDS,
    output_dir=RESULTS_DIR,
    benchmark_name="tier1_axis2_tree_ablation",
    num_steps=num_steps,
    eval_every=eval_every,
    target_class_scenario1=0,
    val_features=val_features,
    val_labels=val_labels,
    config=config,
)

df_axis2 = pd.read_csv(RESULTS_DIR / "tier1_axis2_tree_ablation.csv")
summary_axis2 = df_axis2.groupby("tree_type")[["auprc", "f1", "fpr_at_95_recall"]].agg(["mean", "std"])
print("=== Axis 2: Tree Geometry Ablation Results ===")
print(summary_axis2)


=== Axis 2: Tree Geometry Ablation Results ===
                   auprc                  f1           fpr_at_95_recall  \
                    mean       std      mean       std             mean   
tree_type                                                                 
confusion_ward  0.739791  0.026863  0.605425  0.097864         0.223535   
semantic        0.684816  0.022518  0.523994  0.094018         0.355737   

                          
                     std  
tree_type                 
confusion_ward  0.088945  
semantic        0.169167  


## 4. Scenario 2: Multi-Class Curriculum Benchmark

Evaluates 100-way multi-class classification accuracy, macro-F1, and worst-10-class recall under MCTS curriculum.


In [7]:
s2_seeds = (42, 43, 44, 45, 46) if config.execution_mode == ExecutionMode.REAL else (42,)
s2_steps = 2000 if config.execution_mode == ExecutionMode.REAL else 10
s2_eval = 20 if config.execution_mode == ExecutionMode.REAL else 5

res_s2 = run_benchmark(
    scenario="scenario2",
    train_features=train_features,
    train_labels=train_labels,
    test_features=test_features,
    test_labels=test_labels,
    strategies=("uniform", "mcts_enhanced"),
    tree_types=("semantic",),
    seeds=s2_seeds,
    output_dir=RESULTS_DIR,
    benchmark_name="tier1_scenario2_curriculum",
    num_steps=s2_steps,
    eval_every=s2_eval,
    val_features=val_features,
    val_labels=val_labels,
    config=config,
)

df_s2 = pd.read_csv(RESULTS_DIR / "tier1_scenario2_curriculum.csv")
summary_s2 = df_s2.groupby("strategy")[["top1_accuracy", "macro_f1", "worst_10_class_recall"]].agg(["mean", "std"])
print("=== Scenario 2: Multi-Class Curriculum Results ===")
print(summary_s2)


=== Scenario 2: Multi-Class Curriculum Results ===
              top1_accuracy            macro_f1            \
                       mean       std      mean       std   
strategy                                                    
mcts_enhanced       0.33434  0.015394  0.342589  0.013655   
uniform             0.63854  0.004351  0.637248  0.004391   

              worst_10_class_recall            
                               mean       std  
strategy                                       
mcts_enhanced                0.0068  0.004817  
uniform                      0.3346  0.014536  


## 5. Axis 3: MCTS Hyperparameter Sensitivity Ablation ($c$ and $\alpha$)

Evaluates the sensitivity of MCTS-HNM to the exploration coefficient $c \in \{0.5, 1.0, 2.0, 4.0\}$ and momentum factor $\alpha \in \{0.1, 0.3, 0.5, 0.7\}$.
All other hyperparameters remain fixed. Results are exported to `results/<mode>/tier1_hyperparameter_sensitivity.json` with provenance metadata.


In [8]:
c_vals = (0.5, 1.0, 2.0, 4.0) if config.execution_mode == ExecutionMode.REAL else (0.5, 1.0)
sens_seeds = (42, 43, 44, 45, 46) if config.execution_mode == ExecutionMode.REAL else (42,)
alpha_vals = (0.1, 0.3, 0.5, 0.7) if config.execution_mode == ExecutionMode.REAL else (0.1, 0.3)
sens_steps = 500 if config.execution_mode == ExecutionMode.REAL else 10
sens_eval = 25 if config.execution_mode == ExecutionMode.REAL else 5

res_sens = run_sensitivity_ablation(
    train_features=train_features,
    train_labels=train_labels,
    test_features=test_features,
    test_labels=test_labels,
    c_values=c_vals,
    alpha_values=alpha_vals,
    output_dir=RESULTS_DIR,
    benchmark_name="tier1_hyperparameter_sensitivity",
    num_steps=sens_steps,
    eval_every=sens_eval,
    seeds=sens_seeds,
    config=config,
)

print("=== Axis 3: Hyperparameter Sensitivity Grid Completed ===")
df_sens = pd.DataFrame(res_sens["grid"])
print(df_sens[["c", "alpha", "f1_mean", "auprc_mean"]])


=== Axis 3: Hyperparameter Sensitivity Grid Completed ===
      c  alpha   f1_mean  auprc_mean
0   0.5    0.1  0.808161    0.876656
1   0.5    0.3  0.786590    0.881564
2   0.5    0.5  0.802477    0.888537
3   0.5    0.7  0.760817    0.893676
4   1.0    0.1  0.789181    0.879104
5   1.0    0.3  0.808057    0.888854
6   1.0    0.5  0.764497    0.889128
7   1.0    0.7  0.783935    0.884696
8   2.0    0.1  0.787868    0.886443
9   2.0    0.3  0.783829    0.893347
10  2.0    0.5  0.755101    0.891927
11  2.0    0.7  0.698919    0.887016
12  4.0    0.1  0.761906    0.889532
13  4.0    0.3  0.678442    0.883959
14  4.0    0.5  0.688948    0.886518
15  4.0    0.7  0.574545    0.885024


## 6. Summary & Verification

All Tier-1 multi-seed experimental outputs and visitation diagnostics have been saved to disk under `results/` for figure and table generation in Notebook 05 (`F6`).


In [9]:
print("Exported files in results/:")
for f in sorted(RESULTS_DIR.glob("tier1_*")):
    print(f"  {f.name} ({f.stat().st_size} bytes)")
print("Tier-1 MCTS-HNM benchmark execution complete.")

Exported files in results/:
  tier1_axis1_samplers.csv (3226 bytes)
  tier1_axis1_samplers.csv.meta.json (302 bytes)
  tier1_axis1_samplers.json (249720 bytes)
  tier1_axis1_samplers.json.meta.json (302 bytes)
  tier1_axis2_tree_ablation.csv (1938 bytes)
  tier1_axis2_tree_ablation.csv.meta.json (302 bytes)
  tier1_axis2_tree_ablation.json (232693 bytes)
  tier1_axis2_tree_ablation.json.meta.json (302 bytes)
  tier1_hyperparameter_sensitivity.json (33718 bytes)
  tier1_hyperparameter_sensitivity.json.meta.json (302 bytes)
  tier1_scenario2_curriculum.csv (1249 bytes)
  tier1_scenario2_curriculum.csv.meta.json (302 bytes)
  tier1_scenario2_curriculum.json (591056 bytes)
  tier1_scenario2_curriculum.json.meta.json (302 bytes)
  tier1_visitation_diagnostics.json (461479 bytes)
  tier1_visitation_diagnostics.json.meta.json (302 bytes)
Tier-1 MCTS-HNM benchmark execution complete.
